In [ ]:
import sys
import os
from pathlib import Path

# current_dir=str(Path(os.getcwd()).parent.parent.parent)
# print(f"Current dir: {current_dir}")
# sys.path.insert(0, current_dir)

if __name__ == "__main__":
    root_path = str(Path(os.getcwd()).parents[1])
    print(f"Root path: {root_path}")
    sys.path.insert(0, root_path)
    if __package__ is None:
        __package__ = "comfyui-image-scorer.external_modules.step01ranking"
    print(f"Package: {__package__}")


In [ ]:
import math
from external_modules.step01ranking.cache import get_scored
from external_modules.step01ranking.utils import print_slot_distribution, move_to_slot
from tqdm import tqdm

SLOT_PRECISION = 3

all_images, count = get_scored(limit=100000, comparisons_max=100)
print(f"total images: {len(all_images)}, total comparisons: {count}")

# calculate effective score
for img in all_images:
    img["effective_score"] = int(img["score"]) + float(img["score_modifier"]) * 0.1

# make slots of 0.1 score and then put images in them

Image_bucket = dict[float, list[dict[str, str | float | int | None]]]

image_buckets: Image_bucket = {}
image_buckets_original: Image_bucket = {}
# create empty buckets based on slot precision
for i in range(4 * 10**SLOT_PRECISION + 1):
    bucket = round(1.0 + i / (10**SLOT_PRECISION), SLOT_PRECISION)
    image_buckets[bucket] = []
    image_buckets_original[bucket] = []

print(f"Created {len(image_buckets)} buckets with precision {SLOT_PRECISION}.")

for img in all_images:
    bucket: float = round(float(img["effective_score"]), SLOT_PRECISION)
    if bucket not in image_buckets_original:
        image_buckets_original[bucket] = []
    image_buckets_original[bucket].append(img)

print_slot_distribution(image_buckets_original, precision=SLOT_PRECISION)


for img in all_images:
    bucket: float = round(float(img["effective_score"]), SLOT_PRECISION)
    if bucket not in image_buckets:
        bucket: float = round(float(img["effective_score"]), 0)
    image_buckets[bucket].append(img)


# calculate how many images we want in each bucket to have a more even distribution
total_images = len(all_images)
print(f"total images: {total_images}")

# for precision in range(SLOT_PRECISION + 1):
if SLOT_PRECISION > 0:
    precision = SLOT_PRECISION
    step: float = 1 / (10**precision)
    bucket_count: int = 4 * 10**precision + 1
    target_per_bucket = math.floor(total_images / bucket_count)
    print(f"\ntotal buckets:{bucket_count} ,Target per bucket: {target_per_bucket}")
    # print_slot_distribution(image_buckets, precision=precision)

    # go from 1.0 onwards and reduce images in overfilled slots by moving them to the next slot, until there are no more overfilled slots
    with tqdm(total=len(image_buckets), desc="Redistributing buckets") as pbar:
        for bucket in sorted(image_buckets.keys()):
            higher_slot: float = round(bucket + step, SLOT_PRECISION)
            if higher_slot <= 5.0:
                image_buckets = move_to_slot(
                    image_buckets, bucket, higher_slot, target_per_bucket
                )

            pbar.update(1)
    # print partial results
    print("\nPartial bucket distribution:")
    print_slot_distribution(image_buckets, precision=precision)

    # do the same but backwards from 5.0
    with tqdm(
        total=len(image_buckets), desc="Redistributing buckets (reverse)"
    ) as pbar:
        for bucket in sorted(image_buckets.keys(), reverse=True):
            lower_slot: float = round(bucket - step, SLOT_PRECISION)
            if lower_slot >= 1.0:
                image_buckets = move_to_slot(
                    image_buckets, bucket, lower_slot, target_per_bucket
                )

            pbar.update(1)
    print_slot_distribution(image_buckets, precision=precision)

    # go from 1.0 again but with higher limit
    target_per_bucket = math.ceil(total_images / bucket_count)

    with tqdm(total=len(image_buckets), desc="Redistributing buckets") as pbar:
        for bucket in sorted(image_buckets.keys()):
            higher_slot: float = round(bucket + step, SLOT_PRECISION)
            if higher_slot <= 5.0:
                image_buckets = move_to_slot(
                    image_buckets, bucket, higher_slot, target_per_bucket
                )

            pbar.update(1)

print("\nFinal bucket distribution:")
print_slot_distribution(image_buckets, precision=SLOT_PRECISION)
# larger slot count
larger_slot_count = max(len(image_buckets[bucket]) for bucket in image_buckets)

print(f"\nLarger slot count: {larger_slot_count}")


In [ ]:
# remove elements that dont need to be updated
removed_elements = 0
for bucket in image_buckets:
    previous_len = len(image_buckets[bucket])

    # current_slot=image_buckets[bucket]
    # for img in current_slot:
    #     if round(float(img["effective_score"]), SLOT_PRECISION) == bucket:
    #         del img
    #         removed_elements += 1

    image_buckets[bucket] = [
        img
        for img in image_buckets[bucket]
        if round(float(img["effective_score"]), SLOT_PRECISION) != bucket
    ]

    final_len = len(image_buckets[bucket])
    removed = previous_len - final_len
    if removed > 0:
        removed_elements += removed
        # print(
        #     f"Bucket {bucket}: removed {previous_len-final_len} elements, {final_len} remaining."
        # )

print(f"\nRemoved {removed_elements} elements that didnt need to be updated.")

print_slot_distribution(image_buckets, precision=SLOT_PRECISION)


In [ ]:
from external_modules.step01ranking.cache import update_multiple

with tqdm(total=len(image_buckets), desc="Updating scores") as pbar:
    for bucket in image_buckets:
        path_list: list[str] = [
            str(bucket_img["path"]) for bucket_img in image_buckets[bucket]
        ]

        score = math.floor(bucket)
        score_modifier: float = round(10 * (bucket - score), SLOT_PRECISION - 1)

        if score_modifier > 5:
            score += 1
            score_modifier = round(score_modifier - 10, SLOT_PRECISION)
        if bucket == 2.05:
            print(f"for slot {bucket}, image list:")
            for img in image_buckets[bucket]:
                print(f"score: {img["effective_score"]}")

            print(score, score_modifier)

        # update_multiple(path_list, score, score_modifier)
        pbar.update(1)
